# Evaluation Notebook

This file performs evaluation of the diffusion-based predictor, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the default model directory.
# model_dir = "/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-ensemble3/wandb/run-20250828_140541-nznhbsh7/files"
# model_dir = "/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-obsprob0.5-normalizedObs-wcg0.1/wandb/run-20260306_175250-63u0t05d/files"

# Set the output file.
# output_file = "./results/output_toy_problem_uq.zarr"
output_file = "./results/output_toy_problem_uq2.zarr"

# Set the output file automatically.
# output_file = "./results/diffuser_kf_comparison_drift_" + model_dir.split("/")[-2] + ".zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main

import os
import yaml
import time
from copy import deepcopy
from multiprocessing import Process
os.environ["WANDB__SERVICE_WAIT"] = "300"

In [3]:
def convert_args(argsdict):
    # Convert arguments to the appropriate format.
    arglist = []
    for key, value in argsdict.items():
        if type(value) == dict and "value" in value and not key.startswith("_"):
            v = value["value"]
            if v == None:
                continue
            if type(v) == bool:
                if v:
                    arglist.append(f"--{key}")
                else:
                    arglist.append(f"--no-{key}")
            elif type(v) == list:
                if len(v) == 0:
                    continue
                arglist.append(f"--{key}")
                arglist.extend([str(i) for i in v])
            else:
                arglist.append(f"--{key}")
                arglist.append(str(v))
    return arglist


def wrap_arg_val(val):
    return {"value": val}


def load_args(model_dir):
    # Load arguments from the config file.
    config_file = os.path.join(model_dir, "config.yaml")
    args = yaml.load(open(config_file), Loader=yaml.FullLoader)

    # Set required eval-specific arguments. Do not change these!
    args["use_wandb"] = wrap_arg_val(False)
    args["use_render"] = wrap_arg_val(False)
    args["use_eval"] = wrap_arg_val(True)
    args["model_dir"] = wrap_arg_val(model_dir)
    args["eval_output_file"] = wrap_arg_val(output_file)


    # Set default arguments for evaluation. Feel free to change these!
    args["cuda"] = wrap_arg_val(True)
    args["cuda_idx"] = wrap_arg_val(1)
    args["cuda_idx_predictor"] = args["cuda_idx"]
    args["eval_episodes"] = wrap_arg_val(3)
    args["episode_length"] = wrap_arg_val(200)
    args["seed"] = wrap_arg_val(1)
    args["random_start_positions"] = wrap_arg_val(False)

    # args["disturbance_drift_magnitude"] = wrap_arg_val(0.5)

    return args

# Load arguments for the default model.
# args = load_args(model_dir)

# Evaluation

In [ ]:
# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")
    # print("Output file already exists. Overwriting evaluation.")

else:
# if True:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)
    
    # obs_probabilities = np.arange(0.0, 1.0, 0.01).tolist()
    obs_probabilities = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

    # Add diffuser series.
    for obs_prob in obs_probabilities:
        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-estimate-draft1/wandb/run-20260426_225322-frz5dm0c/files")
        args["eval_series"] = "Estimate"
        args["experiment_name"] = wrap_arg_val(f"uqEst - obs_prob_{obs_prob:.2f}")
        args["observation_probability"] = wrap_arg_val(obs_prob)
        args["diffusion_autoregression_steps"] = wrap_arg_val(1)
        args["random_start_positions"] = wrap_arg_val(False)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-draft1/wandb/run-20260422_010519-83eoz3ta/files")
        args["eval_series"] = "Ensemble"
        args["experiment_name"] = wrap_arg_val(f"uqEnsemble3 - obs_prob_{obs_prob:.2f}")
        args["observation_probability"] = wrap_arg_val(obs_prob)
        args["diffusion_autoregression_steps"] = wrap_arg_val(1)
        args["random_start_positions"] = wrap_arg_val(False)
        start_process(args)
    
    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    MAX_PROCESSES = 30
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.



/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '200', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-observation_mask', '--no-save_checkpoints', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")
/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '200', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-observation_mask', '--no-save_checkpoints', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be Falseu are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be FalsePettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.





/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '200', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-observation_mask', '--no-save_checkpoints', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")
/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '200', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-observation_mask', '--no-save_checkpoints', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be FalsePettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False




...



PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.
PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': 50, 'num_agents': 1, 'num_adversaries': 1, 'num_dimensions': 2, 'world_size': 60.0, 'random_start_positions': False, 'state_per_agent': False, 'observation_probability': 0.8, 'disturbance_drift_magnitude': 0.0, 'adversary_dynamics': 'random', 'lorenz_sigma': 10.0, 'lorenz_rho': 28.0, 'lorenz_beta': 2.6666666666666665}
PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.





/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(


PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': 50, 'num_agents': 1, 'num_adversaries': 1, 'num_dimensions': 2, 'world_size': 60.0, 'random_start_positions': False, 'state_per_agent': False, 'observation_probability': 1.0, 'disturbance_drift_magnitude': 0.0, 'adversary_dynamics': 'random', 'lorenz_sigma': 10.0, 'lorenz_rho': 28.0, 'lorenz_beta': 2.6666666666666665}PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': 50, 'num_agents': 1, 'num_adversaries': 1, 'num_dimensions': 2, 'world_size': 60.0, 'random_start_positions': False, 'state_per_agent': False, 'observation_probability': 0.8, 'disturbance_drift_magnitude': 0.0, 'adversary_dynamics': 'random', 'lorenz_sigma': 10.0, 'lorenz_rho': 28.0, 'lorenz_beta': 2.6666666666666665}PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': 50, 'num_agents': 1, 'num_adversaries': 1, 'num_dimensions': 2, 'world_size': 60.0, 'random_start_positions': False,

/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/t

R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: FalseR_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False

R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action output distributions:Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf

  0: DiagGaussian, action_dim: 2, available_action_dim: infAction output distributions:

  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action ou

# Analysis

In [5]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='toy-problem-draft1', nodes=(Node(di…

In [6]:
import plotly.graph_objects as go
import numpy as np

# Data for plotting
categories = {
    'tracking_error_mean': {
        'title': 'Tracking Error',
        'ylabel': 'Tracking Error (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['tracking_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_trajectory_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_uncertainty_mean': {
        'title': 'Prediction Uncertainty',
        'ylabel': 'Prediction Uncertainty (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_uncertainty_mean']) for experiment in output[series]]
                for series in output
        }
    },
    # 'prediction_uncertainty_error_correlation': {
    #     'title': 'Correlation between Prediction Uncertainty and Prediction Error',
    #     'ylabel': 'Correlation Coefficient',
    #     'xlabel': 'Observation Probability',
    #     # Average over all episodes and all timesteps.
    #     'y': {
    #         series: [np.mean(output[series][experiment]['prediction_uncertainty_error_correlation']) for experiment in output[series]]
    #             for series in output
    #     }
    # },
    'prediction_uncertainty_gap_mean': {
        'title': 'Prediction Uncertainty Gap',
        'ylabel': 'Prediction Uncertainty Gap (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_uncertainty_gap_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_uncertainty_gap_min': {
        'title': 'Prediction Uncertainty Gap (Min)',
        'ylabel': 'Prediction Uncertainty Gap Min (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_uncertainty_gap_min']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_uncertainty_gap_max': {
        'title': 'Prediction Uncertainty Gap (Max)',
        'ylabel': 'Prediction Uncertainty Gap Max (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_uncertainty_gap_max']) for experiment in output[series]]
                for series in output
        }
    }
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in output]

for category in categories:
    fig = go.Figure()
    x = categories[category].get('x', None)
    
    if type(categories[category]['y']) == dict:
        for series, values in categories[category]['y'].items():
            print(f"{category} - {series}: {values}")
            x_local = x
            if x_local is None:
                x_local = np.linspace(0, 1, len(values))
            fig.add_trace(go.Scatter(
                x=x_local,
                y=values,
                mode='lines+markers',
                name=series,
                hovertemplate='%{x:.2f}: %{y:.4f}<extra></extra>'
            ))
    else:
        values = categories[category]['y']
        x_local = x
        if x_local is None:
            x_local = np.linspace(0, 1, len(values))
        fig.add_trace(go.Scatter(
            x=x_local,
            y=values,
            mode='lines+markers',
            hovertemplate='%{x:.2f}: %{y:.4f}<extra></extra>'
        ))
    
    title = categories[category]['title']
    xlabel = 'Scenario' if "xlabel" not in categories[category] else categories[category]['xlabel']
    ylabel = f"Timesteps out of {timesteps}" if "ylabel" not in categories[category] else categories[category]['ylabel']
    
    fig.update_layout(
        # title=f"{title}: avg. over {args['eval_episodes']['value']} episodes, seed {args['seed']['value']}",
        title=f"{title}",
        xaxis_title=xlabel,
        yaxis_title=ylabel,
        hovermode='closest',
        showlegend=True,
        autosize=True,
        height=600,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(
            orientation='h',
            yanchor='top',
            y=-0.2,
            xanchor='center',
            x=0.5
        ),
    )
    
    fig.show()

tracking_error_mean - toy-problem-draft1: [2.5090516, 1.8730288, 0.7923086, 0.5311544, 0.41756374, 0.31618682, 0.29192287, 0.26241726, 0.2540579, 0.23649266, 0.23942398]
tracking_error_mean - toy-problem-estimate-draft1: [3.0924652, 1.8063725, 0.8749837, 0.6469921, 0.40700525, 0.3015526, 0.27655673, 0.26146555, 0.25254872, 0.23095395, 0.22918905]


prediction_error_mean - toy-problem-draft1: [7.90557e-09, 7.369866e-09, 1.06784865e-08, 1.4799405e-08, 1.4106647e-08, 1.6063428e-08, 1.6602044e-08, 1.8550233e-08, 2.1212262e-08, 1.9751724e-08, 2.2136858e-08]
prediction_error_mean - toy-problem-estimate-draft1: [1.1096536e-09, 1.6208463e-09, 2.8570908e-09, 2.535957e-09, 2.8598504e-09, 3.5762893e-09, 3.5368377e-09, 4.6717146e-09, 4.535897e-09, 5.100942e-09, 5.349726e-09]


prediction_trajectory_error_mean - toy-problem-draft1: [7.90557e-09, 7.369866e-09, 1.06784865e-08, 1.4799405e-08, 1.4106647e-08, 1.6063428e-08, 1.6602044e-08, 1.8550233e-08, 2.1212262e-08, 1.9751724e-08, 2.2136858e-08]
prediction_trajectory_error_mean - toy-problem-estimate-draft1: [1.1096536e-09, 1.6208463e-09, 2.8570908e-09, 2.535957e-09, 2.8598504e-09, 3.5762893e-09, 3.5368377e-09, 4.6717146e-09, 4.535897e-09, 5.100942e-09, 5.349726e-09]


prediction_uncertainty_mean - toy-problem-draft1: [0.72816753, 0.29354516, 0.11657477, 0.0971373, 0.07821412, 0.023957638, 0.007848201, 0.0036673062, 0.0024712197, 0.0008988892, 1.0974262e-14]
prediction_uncertainty_mean - toy-problem-estimate-draft1: [0.273803, 0.2771133, 0.29134664, 0.3004177, 0.2999405, 0.29959896, 0.30054677, 0.30088463, 0.301316, 0.3014231, 0.30134568]


prediction_uncertainty_gap_mean - toy-problem-draft1: [0.72816753, 0.29354516, 0.11657476, 0.09713729, 0.0782141, 0.023957621, 0.007848185, 0.003667287, 0.0024711983, 0.00089886953, -2.2136847e-08]
prediction_uncertainty_gap_mean - toy-problem-estimate-draft1: [0.273803, 0.2771133, 0.29134664, 0.3004177, 0.2999405, 0.29959896, 0.30054677, 0.3008846, 0.301316, 0.30142307, 0.30134568]


prediction_uncertainty_gap_min - toy-problem-draft1: [-6.1358634e-08, -5.724219e-08, -7.420777e-08, -1.03013576e-07, -9.761811e-08, -1.11802144e-07, -1.08457144e-07, -1.1257825e-07, -1.3441468e-07, -1.17178985e-07, -1.2428342e-07]
prediction_uncertainty_gap_min - toy-problem-estimate-draft1: [0.066086955, 0.07587574, 0.079042956, 0.078764066, 0.08046896, 0.08076428, 0.08116472, 0.08140859, 0.08150908, 0.08162617, 0.08156631]


prediction_uncertainty_gap_max - toy-problem-draft1: [4.911409, 1.9196337, 0.8041435, 0.6126106, 0.47110358, 0.15589762, 0.052899558, 0.024771817, 0.016983487, 0.0056136353, 4.7972985e-14]
prediction_uncertainty_gap_max - toy-problem-estimate-draft1: [1.037609, 1.0651902, 1.0677081, 1.083907, 1.066194, 1.065074, 1.0662531, 1.0665519, 1.0664592, 1.0672534, 1.0672463]
